# 1. Import thư viện

In [1]:
!pip install -r requirment.txt

   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/409.8 kB ? eta -:--:--
   ---

In [1]:
"""
Các dòng code bên dưới dùng để import các thư viện thiết yếu để xây dựng mô hình RAG
"""
import json
import sys
import uuid
import os
import time
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.documents import Document
from underthesea import word_tokenize
from tqdm import tqdm
from langchain_ollama import ChatOllama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import RedisChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks import BaseCallbackHandler

In [2]:
import json

# Hàm nạp dữ liệu từ file JSON
def load_layer_b(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# Nạp 2 bộ tri thức Nam và Nữ
# Lưu ý: Sửa lại đường dẫn file cho đúng với folder của bạn
layer_b_female = load_layer_b("Layer_B_Female_Knowledge.json")
layer_b_male = load_layer_b("Layer_B_Male_Knowledge.json")

print(f"Đã nạp {len(layer_b_female)} quy tắc Nữ và {len(layer_b_male)} quy tắc Nam.")

Đã nạp 880 quy tắc Nữ và 416 quy tắc Nam.


In [3]:
# ═══════════════════════════════════════════════════════════════════
# LAYER B – BƯỚC 1: Tìm công thức phối đồ cho sản phẩm của user
# ═══════════════════════════════════════════════════════════════════

def find_matching_rule(user_query: str, gender: str = "female") -> dict | None:
    """
    Tìm rule trong Layer B khớp với sản phẩm mà user đang mô tả.
    Cách hoạt động: Đếm số từ khớp giữa câu hỏi của user và rule_key,
    trả về rule có điểm cao nhất.
    """
    knowledge = layer_b_female if gender == "female" else layer_b_male
    query_lower = user_query.lower()

    best_rule  = None
    best_score = 0

    for rule in knowledge:
        # Lấy phần product type sau dấu "|"  Ví dụ: "basic T-shirt"
        parts        = rule["rule_key"].split("|")
        category_str = parts[0].strip().lower()  # "áo mặc trong (áo thun/sơ mi)"
        product_str  = parts[1].strip().lower() if len(parts) > 1 else ""

        # Gộp cả category lẫn product_type để so khớp
        all_keywords = (category_str + " " + product_str).split()

        score = sum(1 for kw in all_keywords if kw in query_lower)
        if score > best_score:
            best_score = score
            best_rule  = rule

    if best_score == 0:
        return None
    return best_rule


# ═══════════════════════════════════════════════════════════════════
# LAYER B – BƯỚC 2: Tìm rule chi tiết cho TỪNG món đồ gợi ý
# ═══════════════════════════════════════════════════════════════════

def find_outfit_details(base_rule: dict, gender: str = "female") -> dict:
    """
    Với mỗi category trong goi_y_phoi_cung của base_rule,
    tìm rule cụ thể có CÙNG phong_cach + boi_canh trong Layer B.

    Trả về dict: { "Giày dép": rule_chi_tiet, "Túi xách": rule_chi_tiet, ... }
    """
    knowledge   = layer_b_female if gender == "female" else layer_b_male
    outfit_rules = {}

    for category in base_rule.get("goi_y_phoi_cung", []):
        matched = [
            r for r in knowledge
            if r["rule_key"].startswith(category)
            and r["phong_cach"] == base_rule["phong_cach"]
            and r["boi_canh"]   == base_rule["boi_canh"]
        ]
        if matched:
            outfit_rules[category] = matched[0]

    return outfit_rules


# ── Chạy thử ──────────────────────────────────────────────────────
test_rule = find_matching_rule("áo sơ mi công sở", gender="female")
if test_rule:
    print(f"✅ Tìm thấy: {test_rule['rule_key']}")
    print(f"   Phong cách : {test_rule['phong_cach']}")
    print(f"   Bối cảnh   : {test_rule['boi_canh']}")
    print(f"   Phối với   : {test_rule['goi_y_phoi_cung']}")

    details = find_outfit_details(test_rule, gender="female")
    print(f"\n📋 Chi tiết outfit ({len(details)} món):")
    for cat, rule in details.items():
        product_type = rule["rule_key"].split("|")[1].strip()
        print(f"   {cat:30s} → {product_type}")


✅ Tìm thấy: Áo khoác ngoài | soft lamb wool
   Phong cách : Sophisticated urban
   Bối cảnh   : Mùa đông – Học đường
   Phối với   : ['Áo khoác ngoài', 'Áo mặc trong (áo thun/sơ mi)', 'Quần/Chân váy', 'Phụ kiện', 'Túi xách', 'Giày dép']

📋 Chi tiết outfit (6 món):
   Áo khoác ngoài                 → soft lamb wool
   Áo mặc trong (áo thun/sơ mi)   → basic shirt
   Quần/Chân váy                  → preppy A-line skirt
   Phụ kiện                       → wool beret
   Túi xách                       → quilted chain bag
   Giày dép                       → platform loafers


# 2. Chunking các metadata của sản phẩm thời trang

In [4]:
def process_fashion_metadata(file_path):
    """
    Hàm này được sử dụng để đọc file JSONL và chuyển đổi thành danh sách các LangChain Documents
    """
    documents = []

    # Mở và đọc từng dòng trong file JSONL
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): # Kiểm tra các dòng trống và bỏ qua nó
                continue
            item = json.loads(line) # Load từng dòng json

            # Bước 1: Xử lý metadata sản phẩm để lấy ra các trường quan trọng và đặt nó cùng với "Bước 2"
            # Trích xuất danh sách link ảnh từ mảng images
            image_urls = [img.get("large") for img in item.get("images", []) if "large" in img]

            # Lấy các thông tin metadata của sản phẩm 
            metadata = {
                "product_id": item.get("product_id", ""),
                "category": item.get("category", ""),
                "department": item.get("department", ""),
                "brand": item.get("brand", ""),
                "price": item.get("price", 0),
                "images": image_urls
            }

            # Bước 2: Xử lý theo hướng văn bản hóa dữ liệu có cấu trúc
            # Lấy thông tin details từ metadata của sản phẩm
            details = item.get("details", {})
            main_color = details.get("main_color", "Không có thông tin về màu sắc")
            material = details.get("material", "Không có thông tin về chất liệu")
            size = details.get("size", "Không có thông tin về kích thước")
            pattern = details.get("pattern", "Không có thông tin về họa tiết")
            
            # Khung văn bản mô tả sản phẩm
            page_content = (
                f"Sản phẩm {item.get('title', 'Không có tên sản phẩm')} thuộc danh mục {item.get('category', '')} "
                f"dành cho {item.get('department', '')}. "
                f"Thương hiệu: {item.get('brand', 'Không có thông tin về thương hiệu')}. "
                f"Mức giá: {item.get('price', 0)} VNĐ. "
                f"Đặc điểm chi tiết: có các loại màu sắc: {main_color}, chất liệu: {material}, kích cỡ: {size}, họa tiết: {pattern}. "
                f"Phù hợp sử dụng cho {item.get('season', '')} và các dịp {item.get('occasion', '')}. "
                f"Mô tả chi tiết: {item.get('description', '')}"
            )
            
            # Bước 3: Tạo LangChain Documents
            doc = Document(page_content=page_content, metadata=metadata)
            documents.append(doc)
            """
            Sau khi tạo xong LangChain Documents thì đối tượng Document nó sẽ có 2 thành phần sau:
            - page_content (string): Đây chính là phần văn bản đã được "văn bản hóa" từ dữ liệu thô có cấu trúc ở "Bước 2". Phần này sau đó sẽ
              đi qua một mô hình Embedding để tạo vector ngữ nghĩa.
            - metadata (dict): Đây là một từ điển chứa các thông tin bổ trợ. Phần này sẽ được giữ nguyên cấu trúc của nó.
            """

        return documents

def process_all_directory(directory_path):
    """
    Hàm lặp qua tất cả các file .jsonl trong một thư mục và gộp data lại.
    """
    all_documents = []
    
    # Duyệt qua tất cả các file có trong thư mục
    for filename in sorted(os.listdir(directory_path)):
        # Chỉ xử lý những file có đuôi là .jsonl
        if filename.endswith(".jsonl"):
            # Tạo đường dẫn tuyệt đối cho từng file
            file_path = os.path.join(directory_path, filename)
            
            print(f"[THÔNG BÁO] Đang đọc file: {filename}...")
            
            # Gọi lại hàm process_fashion_metadata(file_path) đã được định nghĩa ở phía trên
            docs = process_fashion_metadata(file_path)
            
            # LƯU Ý QUAN TRỌNG: Dùng .extend() để gộp các list lại với nhau
            # Không dùng .append() vì nó sẽ tạo ra list lồng list
            all_documents.extend(docs)
            
    return all_documents

# 3. Sử dụng mô hình Embedding để xử lý các Chunks vào lưu vào Qdrant vector DB

In [ ]:
# class VietnameseSegmentedEmbeddings(Embeddings):
#     # Hàm này dùng để khởi tạo mô hình Embedding
#     def __init__(self, model_name="VoVanPhuc/sup-SimCSE-VietNamese-phobert-base"):
#         self.hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)
        
#     # Hàm này nhận đầu vào là text và dùng word_tokenize của underthesea để thực hiện tách từ và trả về từ đã tách
#     def _segment_text(self, text: str) -> str:
#         if not text:
#             return ""
#         return word_tokenize(text, format="text")

#     # Hàm này sử dụng mô hình embedding đã khởi tạo để vector hóa dữ liệu
#     def embed_documents(self, texts: list[str]) -> list[list[float]]:
#         segmented_texts = [self._segment_text(text) for text in texts]
#         return self.hf_embeddings.embed_documents(segmented_texts)

#     # Hàm này sử dụng mô hình embedding đã khởi tọa để vector hóa câu truy vấn của người dùng
#     def embed_query(self, text: str) -> list[float]:
#         segmented_text = self._segment_text(text)
#         return self.hf_embeddings.embed_query(segmented_text)

In [5]:
# Hướng tiếp cận mới: Sử dụng BGE-M3 (Không cần tách từ thủ công)
class BGEM3Embeddings(Embeddings):
    def __init__(self, model_name="BAAI/bge-m3"):
        # BGE-M3 thường cho ra vector size 1024 (lớn hơn PhoBERT là 768)
        self.hf_embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cpu'} # Chuyển thành 'cuda' nếu bạn có GPU
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        # Không dùng word_tokenize, đẩy trực tiếp text thô
        return self.hf_embeddings.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        return self.hf_embeddings.embed_query(text)

In [6]:
"""
Đoạn code bên dưới sẽ hoạt động theo các bước như sau:
- Bước 1: Trỏ đến thư mục chứa các metadata sản phẩm thời trang dưới dạng .jsonl, sau đó sẽ quét toàn bộ file và thực hiện chunking từng file một.
- Bước 2: Sau khi setup thành công Qdrant trên Docker, thì kết nối tới Qdrant đó thông qua địa chỉ ip local được cung cấp. Sau đó, tạo một collections 
          mới và thiết lập thông số cho vectors được lưu.
- Bước 3: Tiếp tục khởi tạo một VectorStore. VectorStore có nhiệm vụ số hóa các chunk tài liệu và lưu nó vào trong vector database đã được thiết lập.
"""
if __name__ == "__main__":
    # Trỏ đến thư mục chứa tất cả các file JSONL của bạn
    folder_path = "./Fashion_Metadata"

    # Khởi tạo Custom Embedding
    custom_embeddings = BGEM3Embeddings()

    if os.path.exists(folder_path):
        print(f"[THÔNG BÁO] Bắt đầu quét thư mục: {folder_path}\n" + "="*50)
        
        # Gọi hàm xử lý toàn bộ
        all_docs = process_all_directory(folder_path)
        
        print("="*50)
        print(f"[THÔNG BÁO] HOÀN TẤT! Tổng số lượng chunk (documents) thu được: {len(all_docs)}")

        qdrant_url = "http://localhost:6333"
        collection_name = "fashion_products_bge_m3"
        
        print(f"[THÔNG BÁO] Bắt đầu đẩy dữ liệu vào Qdrant tại: {qdrant_url}")
        
        client = QdrantClient(url=qdrant_url)
        
        # 1. KIỂM TRA SỐ LƯỢNG ĐÃ LƯU TRONG DATABASE
        if not client.collection_exists(collection_name):
            print(f"[THÔNG BÁO] Tạo mới collection '{collection_name}'...")
            client.create_collection(
                collection_name=collection_name,
                vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
            )
            current_count = 0 # Chưa có gì
        else:
            # Lấy thông tin database hiện tại
            collection_info = client.get_collection(collection_name)
            current_count = collection_info.points_count # Số lượng vector đang có sẵn
            print(f"[THÔNG BÁO] Tìm thấy {current_count} sản phẩm đã được lưu trước đó.")
        
        # 2. CẮT BỎ PHẦN DỮ LIỆU ĐÃ LÀM XONG
        # Lấy từ vị trí current_count cho đến hết
        remaining_docs = all_docs[current_count:]
        
        if len(remaining_docs) == 0:
            print("\n[THÔNG BÁO] 🎉 Toàn bộ dữ liệu đã được xử lý xong từ trước. Không cần chạy lại!")
        else:
            print(f"[THÔNG BÁO] Bắt đầu nhúng và lưu {len(remaining_docs)} sản phẩm còn lại...")
            
            # Khởi tạo VectorStore
            vector_db = QdrantVectorStore(
                client=client,
                collection_name=collection_name,
                embedding=custom_embeddings,
            )
            
            batch_size = 128
            
            # 3. CHẠY THANH TIẾN TRÌNH CHO PHẦN CÒN LẠI
            # Tham số `initial` giúp thanh tiến trình không bắt đầu từ 0, mà hiển thị đúng phần trăm thực tế
            with tqdm(total=len(all_docs), initial=current_count, desc="Tiến độ Vector hóa", unit="SP") as pbar:
                for i in range(0, len(remaining_docs), batch_size):
                    # Cắt lô dữ liệu tiếp theo
                    batch = remaining_docs[i : i + batch_size]
                    
                    # Đẩy vào Qdrant
                    vector_db.add_documents(documents=batch)
                    
                    # Cập nhật thanh tiến trình
                    pbar.update(len(batch))
        
            print("\n[THÔNG BÁO] 🎉 ĐÃ LƯU HOÀN TẤT VÀO VECTOR DATABASE!")
        
    else:
        print(f"[THÔNG BÁO] Không tìm thấy thư mục {folder_path}. Vui lòng kiểm tra lại.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

KeyboardInterrupt: 

# 4. Xây dựng Retriever để truy hồi dữ liệu

In [6]:
def load_vector_db():
    print("[THÔNG BÁO] Đang khởi tạo mô hình Embedding...")
    custom_embeddings = BGEM3Embeddings()
    
    print("[THÔNG BÁO] Đang kết nối tới Qdrant Database...")
    # qdrant_url = "http://localhost:6333"
    # client = QdrantClient(url=qdrant_url)
    client = QdrantClient(path="./qdrant_data")
    # Khởi tạo QdrantVectorStore và gán vào biến vector_db
    vector_db = QdrantVectorStore(
        client=client,
        collection_name="fashion_products_bge_m3",
        embedding=custom_embeddings
    )
    
    print("✅ Đã kết nối thành công!")
    return vector_db

# Sử dụng trong file Chatbot
if __name__ == "__main__":
    vector_db = load_vector_db()

    # Thiết lập retriever để truy hồi thông tin
    retriever = vector_db.as_retriever(
        search_type="similarity_score_threshold", # Tìm kiếm theo độ tương đồng Vector dựa trên một ngưỡng
        search_kwargs={"k": 5, # "k" là số lượng kết quả bạn muốn lấy ra
                       "score_threshold": 0.7 # Chỉ lấy ra các sản phẩm có độ tương đồng Vector từ 70% trở lên
                      }    
    )
    
    print("[THÔNG BÁO] Đã khởi tạo Retriever thành công!")

[THÔNG BÁO] Đang khởi tạo mô hình Embedding...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[THÔNG BÁO] Đang kết nối tới Qdrant Database...


C:\Users\DELL\AppData\Local\Temp\ipykernel_27212\1350268014.py:8: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <fashion_products_bge_m3> contains 48938 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path="./qdrant_data")


✅ Đã kết nối thành công!
[THÔNG BÁO] Đã khởi tạo Retriever thành công!


# Layer B


In [7]:
# ═══════════════════════════════════════════════════════════════════
# LAYER B → LAYER A: Bảng ánh xạ và Keyword Router cho Phụ kiện
# ═══════════════════════════════════════════════════════════════════

# Ánh xạ 7/8 category Layer B → danh mục trong metadata.category của Layer A
CATEGORY_MAPPING = {
    "Áo mặc trong (áo thun/sơ mi)": ["Áo"],
    "Áo khoác ngoài"               : ["Áo khoác"],
    "Áo khoác nhẹ/Áo len"          : ["Áo khoác"],
    "Quần/Chân váy"                : ["Quần", "Chân váy"],
    "Đầm/Jumpsuit"                 : ["Đầm", "Jumpsuit"],
    "Giày dép"                     : ["Giày"],
    "Túi xách"                     : ["Túi xách"],
    "Phụ kiện"                     : None,  # Xử lý riêng qua keyword router bên dưới
}

# Keyword router cho "Phụ kiện" → đúng danh mục Layer A
# (vì 1 category "Phụ kiện" trong Layer B có thể là Mũ, Đồng hồ, Nhẫn, ...)
PHU_KIEN_KEYWORD_ROUTER = {
    "Mũ"            : ["beret", "hat", "cap", "beanie", "fedora", "bucket",
                        "brim", "flat cap", "earflap", "snood", "balaclava", "trapper"],
    "Găng tay"      : ["gloves", "glove", "arm warmer"],
    "Kính mắt"      : ["glasses", "sunglasses", "sunglass"],
    "Đồng hồ"       : ["watch"],
    "Dây chuyền"    : ["necklace", "chain pendant", "chain"],
    "Bông tai"      : ["earring", "earrings"],
    "Vòng tay"      : ["bracelet"],
    "Nhẫn"          : ["ring"],
    "Ghim cài áo"   : ["brooch", "pin", "badge"],
    # Fallback: tất, thắt lưng, khăn, băng đô...
    "Phụ kiện hỗ trợ": ["socks", "sock", "scarf", "tie", "belt",
                         "bandana", "headband", "suspender"],
}

def get_layer_a_categories(layer_b_category: str, product_type: str) -> list[str]:
    """
    Trả về danh sách tên category Layer A cần filter khi tìm Qdrant.
    - 7 category đơn giản → tra CATEGORY_MAPPING.
    - "Phụ kiện"          → dùng PHU_KIEN_KEYWORD_ROUTER để tìm đúng category.
    """
    if layer_b_category != "Phụ kiện":
        return CATEGORY_MAPPING.get(layer_b_category, [])

    # Keyword router cho Phụ kiện
    ptype_lower = product_type.lower()
    for layer_a_cat, keywords in PHU_KIEN_KEYWORD_ROUTER.items():
        if any(kw in ptype_lower for kw in keywords):
            return [layer_a_cat]

    return ["Phụ kiện hỗ trợ"]  # fallback nếu không khớp keyword nào


# ── Kiểm tra nhanh ────────────────────────────────────────────────
test_cases = [
    ("Áo mặc trong (áo thun/sơ mi)", "basic T-shirt"),
    ("Giày dép",                      "platform loafers"),
    ("Phụ kiện",                      "wool beret"),
    ("Phụ kiện",                      "quilted chain bag"),   # ← noisy data
    ("Phụ kiện",                      "color-block argyle socks"),
    ("Quần/Chân váy",                 "preppy A-line skirt"),
]
print("Kiểm tra get_layer_a_categories():")
for cat_b, ptype in test_cases:
    result = get_layer_a_categories(cat_b, ptype)
    print(f"  ({cat_b:35s} | {ptype:30s}) → {result}")


Kiểm tra get_layer_a_categories():
  (Áo mặc trong (áo thun/sơ mi)        | basic T-shirt                 ) → ['Áo']
  (Giày dép                            | platform loafers              ) → ['Giày']
  (Phụ kiện                            | wool beret                    ) → ['Mũ']
  (Phụ kiện                            | quilted chain bag             ) → ['Dây chuyền']
  (Phụ kiện                            | color-block argyle socks      ) → ['Phụ kiện hỗ trợ']
  (Quần/Chân váy                       | preppy A-line skirt           ) → ['Quần', 'Chân váy']


In [8]:
from qdrant_client.http.models import Filter, FieldCondition, MatchAny

def get_products_for_outfit(
    product_type: str,
    layer_b_category: str,
    phong_cach: str,
    vector_db
) -> list:
    """
    Tìm sản phẩm thật trong Layer A (Qdrant) cho 1 món đồ trong outfit.

    Parameters:
        product_type    : Loại sản phẩm cụ thể từ rule_key Layer B
                          Ví dụ: "platform loafers", "preppy A-line skirt"
        layer_b_category: Category Layer B  Ví dụ: "Giày dép", "Phụ kiện"
        phong_cach      : Phong cách từ Layer B  Ví dụ: "Japanese Cityboy"
        vector_db       : QdrantVectorStore đã khởi tạo
    """
    # Lấy danh mục Layer A tương ứng
    target_categories = get_layer_a_categories(layer_b_category, product_type)

    # Tạo filter metadata.category – MatchAny để hỗ trợ nhiều category
    # Ví dụ: "Quần/Chân váy" → filter category IN ["Quần", "Chân váy"]
    search_filter = None
    if target_categories:
        search_filter = Filter(
            must=[
                FieldCondition(
                    key="metadata.category",
                    match=MatchAny(any=target_categories)
                )
            ]
        )

    # Query giàu ngữ cảnh: product type + phong cách
    query = f"{product_type} {phong_cach}"

    results = vector_db.similarity_search(
        query=query,
        k=3,
        filter=search_filter
    )
    return results


# 5. Khởi tạo LLM để tích hợp vào hệ thống Chatbot

In [9]:
print("[THÔNG BÁO] Đang khởi tạo mô hình Qwen local...")
llm = ChatOllama(
    # model="qwen3:4b-instruct", 
    model="qwen2.5:3b-instruct",
    temperature=0.4,
    timeout=120,
    num_predict=350,
    num_ctx=8192,
)
print("[THÔNG BÁO] Đã khởi tạo LLM thành công!")

[THÔNG BÁO] Đang khởi tạo mô hình Qwen local...
[THÔNG BÁO] Đã khởi tạo LLM thành công!


# 6. Cấu hình Prompt và Redis

In [10]:
# 1. Kịch bản chính cho Chatbot (Chống Bịa Đặt & Trích Xuất Chính Xác)
system_prompt = """Bạn là chuyên viên tư vấn thời trang cao cấp của shop.

QUY TẮC TỐI CAO (CHỐNG BỊA ĐẶT - ANTI-HALLUCINATION):
1. CHỈ SỬ DỤNG thông tin có trong phần "DỮ LIỆU SẢN PHẨM" bên dưới để trả lời.
2. TUYỆT ĐỐI KHÔNG tự bịa ra tên sản phẩm, không bịa ID, giá tiền, màu sắc hay chất liệu. 
3. NẾU KHÔNG TÌM THẤY SẢN PHẨM KHỚP NHU CẦU: Khẳng định ngay là shop chưa có mẫu này, xin lỗi và mời khách xem mẫu khác. Không được cố gắng đoán hoặc bịa ra sản phẩm.

QUY TẮC TRÍCH XUẤT VÀ ĐỊNH DẠNG:
4. BẮT BUỘC TRÍCH XUẤT CHÍNH XÁC 100%: Khi giới thiệu một sản phẩm, bạn PHẢI copy y nguyên trường [Sản phẩm] và [MÃ_SP] từ dữ liệu. Không thêm, không bớt, không dịch thuật.
5. TRÌNH BÀY BẮT BUỘC theo khuôn mẫu sau:
   - **[TÊN_SP]** (Mã SP: [MÃ_SP])
   - Giá: [GIÁ_TIỀN] VNĐ
   - Đặc điểm chi tiết: Lấy ra các thông tin về đặc điểm chi tiết của sản phẩm (màu sắc, chất liệu, kích cỡ, họa tiết) và dựa vào nó để VIẾT THÀNH MỘT CÂU HOÀN CHỈNH, KHÔNG ĐƯỢC BỊA ĐẶT THÔNG TIN.
   - (Viết 1-2 câu tư vấn thân thiện, khen ngợi sản phẩm dựa trên đặc điểm và mô tả chi tiết của nó)
6. GIỚI HẠN: Trả lời tự nhiên, thân thiện nhưng không vượt quá 200 từ.

DỮ LIỆU SẢN PHẨM:
{context}"""

QA_PROMPT = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"), 
    ("human", "{input}")
])

REDIS_URL = "redis://localhost:6379"

def get_message_history(session_id: str):
    history = RedisChatMessageHistory(session_id, url=REDIS_URL)
    current_messages = history.messages
    if len(current_messages) > 6: 
        kept_messages = current_messages[-6:]
        history.clear()
        history.add_messages(kept_messages)
    return history

print("[THÔNG BÁO] Đã cấu hình Kịch bản chuẩn ChatML và Redis thành công!")

[THÔNG BÁO] Đã cấu hình Kịch bản chuẩn ChatML và Redis thành công!


# 7. Xây dựng kiến trúc RAG

In [11]:
print("[THÔNG BÁO] Đang lắp ráp Pipeline Retriever Thông minh...")

# 1. KỊCH BẢN PHỤ: Dạy AI cách viết lại câu hỏi
contextualize_q_system_prompt = """BẠN LÀ MỘT CÔNG CỤ VIẾT LẠI CÂU TRUY VẤN, BẠN KHÔNG PHẢI LÀ CHATBOT.
Nhiệm vụ: Đọc "Lịch sử trò chuyện" và "Câu hỏi mới", VIẾT nó thành MỘT CÂU TRUY VẤN HOÀN CHỈNH.

QUY TẮC TUYỆT ĐỐI:
1. CHỈ IN RA CÂU TRUY VẤN HOÀN CHỈNH. TUYỆT ĐỐI không thực hiện những nhiệm vụ khác mà không liên quan tới VAI TRÒ của bạn.
2. Nếu đầu vào KHÔNG PHẢI CÂU TRUY VẤN của người dùng thì TRẢ VỀ CÂU TRUY VẤN GỐC.
3. Nếu đầu vào LÀ CÂU TRUY VẤN của người dùng, nhưng bạn KHÔNG CHẮC CHẮN trong việc viết lại câu truy vấn thì TRẢ VỀ CÂU TRUY VẤN GỐC.

VÍ DỤ BẮT BUỘC TUÂN THỦ:
- Khách: Có áo sơ mi trắng nam không? -> Kết quả: Cửa hàng có bán áo sơ mi trắng nam không?
- Lịch sử: Khách vừa hỏi về "Shop có áo thun nam màu đỏ không". Khách hỏi tiếp: Có size XL không? -> Kết quả: Áo thun nam màu đỏ mà Shop bán có size XXL không?
"""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

document_prompt_template = """
[MÃ_SP: {product_id}]
THÔNG TIN CHI TIẾT: {page_content}
"""
doc_prompt = PromptTemplate.from_template(document_prompt_template)

# 2. KHỞI TẠO CÁC CHUỖI RAG
# Chuỗi 1: Trợ lý viết lại câu hỏi
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# Chuỗi 2: Nhồi dữ liệu từ Qdrant vào Kịch bản chính (QA_PROMPT ở Cell 6)
document_chain = create_stuff_documents_chain(llm=llm, prompt=QA_PROMPT, document_prompt=doc_prompt)

# Chuỗi 3: Ghép nối Trợ lý viết lại câu vào hệ thống
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)

# 4. TRÙM REDIS ĐỂ LƯU LỊCH SỬ
full_chat_chain = RunnableWithMessageHistory(
    rag_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

print("[THÔNG BÁO] Hệ thống Retriever Thông minh đã sẵn sàng!")

[THÔNG BÁO] Đang lắp ráp Pipeline Retriever Thông minh...
[THÔNG BÁO] Hệ thống Retriever Thông minh đã sẵn sàng!


# 8. Cấu hình Intent Detection + Outfit Chain (Layer B)

In [12]:
# ═══════════════════════════════════════════════════════════════════
# INTENT DETECTION: phân biệt user đang hỏi phối đồ hay tìm sản phẩm
# ═══════════════════════════════════════════════════════════════════

OUTFIT_KEYWORDS = [
    "phối", "phối đồ", "mặc với", "kết hợp", "outfit", "mix",
    "mặc gì", "đi với", "hợp với", "bộ đồ", "mặc cùng", "style với",
]
MALE_KEYWORDS   = ["nam", "con trai", "anh", "bạn trai", "chàng", "đàn ông"]

def detect_intent(query: str) -> str:
    """Trả về 'outfit' nếu user hỏi phối đồ, ngược lại trả về 'search'."""
    q = query.lower()
    return "outfit" if any(kw in q for kw in OUTFIT_KEYWORDS) else "search"

def detect_gender(query: str) -> str:
    """Trả về 'male' nếu câu hỏi có từ chỉ giới tính nam, mặc định 'female'."""
    return "male" if any(kw in query.lower() for kw in MALE_KEYWORDS) else "female"


# ═══════════════════════════════════════════════════════════════════
# OUTFIT CHAIN: chain riêng cho luồng phối đồ (không qua RAG retriever)
# ═══════════════════════════════════════════════════════════════════

OUTFIT_SYSTEM_PROMPT = """Bạn là chuyên viên tư vấn thời trang cao cấp của shop.

NHIỆM VỤ: Dựa vào "CÔNG THỨC PHỐI ĐỒ" và "SẢN PHẨM GỢI Ý" bên dưới,
tư vấn cho khách một bộ outfit hoàn chỉnh.

QUY TẮC:
1. CHỈ giới thiệu sản phẩm có trong phần "SẢN PHẨM GỢI Ý". KHÔNG tự bịa.
2. Với mỗi món đồ, trình bày theo khuôn mẫu:
   • **[TÊN SẢN PHẨM]** (Mã SP: [MÃ_SP]) – [GIÁ] VNĐ
     → [1 câu mô tả + lý do phù hợp với outfit]
3. Kết thúc bằng 1-2 câu tổng kết về phong cách tổng thể của bộ đồ.
4. Giới hạn: không quá 250 từ. Giọng văn thân thiện, chuyên nghiệp.

{outfit_context}"""

outfit_prompt = ChatPromptTemplate.from_messages([
    ("system", OUTFIT_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Chain đơn giản: prompt | llm (không có retriever)
outfit_llm_chain = outfit_prompt | llm

# Bọc Redis history vào – dùng CÙNG get_message_history với full_chat_chain
outfit_chain_with_history = RunnableWithMessageHistory(
    outfit_llm_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("[THÔNG BÁO] Đã cấu hình Outfit Chain thành công!")


[THÔNG BÁO] Đã cấu hình Outfit Chain thành công!


# 8. Luồng chạy chính

In [15]:
SESSION_ID = str(uuid.uuid4())

class SpyRetrieverHandler(BaseCallbackHandler):
    """Hiển thị câu hỏi sau khi đã được viết lại bởi LLM."""
    def on_retriever_start(self, serialized: dict, query: str, **kwargs):
        print(f"\n🕵️  [AI ĐÃ VIẾT LẠI CÂU HỎI THÀNH]: {query}\n")

print("=" * 60)
print("  👗👔  CHATBOT TƯ VẤN THỜI TRANG ĐÃ SẴN SÀNG  👔👗  ")
print("        (Nhập '0' rồi Enter để kết thúc)         ")
print("=" * 60 + "\n")

while True:
    user_input = input("👤 Bạn: ")

    if user_input.strip() == "0":
        print("\n🤖 Chatbot: Dạ, cảm ơn bạn đã ghé thăm shop. Hẹn gặp lại bạn nhé!")
        break

    if not user_input.strip():
        continue

    # ── Phát hiện intent ──────────────────────────────────────────
    intent = detect_intent(user_input)
    gender = detect_gender(user_input)
    print(f"🔍 [Intent: {intent.upper()} | Gender: {gender}]")

    print("🤖 Chatbot: ", end="")
    start_time       = time.time()
    first_token_time = None

    try:
        if intent == "outfit":
            # ── LUỒNG PHỐI ĐỒ: Layer B → Layer A → Outfit Chain ──
            outfit_context = build_outfit_context(user_input, gender)

            for chunk in outfit_chain_with_history.stream(
                {
                    "input"          : user_input,
                    "outfit_context" : outfit_context,
                },
                config={"configurable": {"session_id": SESSION_ID}},
            ):
                token = chunk.content if hasattr(chunk, "content") else str(chunk)
                if token:
                    if first_token_time is None:
                        first_token_time = time.time()
                    print(token, end="", flush=True)

        else:
            # ── LUỒNG TÌM SẢN PHẨM: Pipeline RAG gốc ─────────────
            for chunk in full_chat_chain.stream(
                {"input": user_input},
                config={
                    "configurable": {"session_id": SESSION_ID},
                    "callbacks"   : [SpyRetrieverHandler()],
                },
            ):
                if "answer" in chunk:
                    if first_token_time is None:
                        first_token_time = time.time()
                    print(chunk["answer"], end="", flush=True)

        end_time = time.time()
        if first_token_time is None:
            first_token_time = end_time

        ttft       = first_token_time - start_time
        total_time = end_time - start_time

        print(f"\n\n⏱️  [THỐNG KÊ TỐC ĐỘ]:")
        print(f"   - Nhả chữ đầu tiên sau : {ttft:.2f} giây")
        print(f"   - Hoàn thành toàn bộ sau: {total_time:.2f} giây")

    except Exception as e:
        print(f"\n[LỖI HỆ THỐNG] Đã xảy ra sự cố: {e}")

    print("\n\n" + "-" * 60 + "\n")


  👗👔  CHATBOT TƯ VẤN THỜI TRANG ĐÃ SẴN SÀNG  👔👗  
        (Nhập '0' rồi Enter để kết thúc)         

🔍 [Intent: OUTFIT | Gender: female]
🤖 Chatbot: Dựa theo yêu cầu của bạn, tôi có một gợi ý cho bộ outfit hoàn chỉnh:

**Đầm dự tiệc Lady Dazzle (Mã SP: LD123) – [GIÁ] VNĐ**
→ Đầm này mang phong cách thời trang nữ tính nhưng vẫn giữ được sự chỉn chu và tin cậy, phù hợp với buổi ra mắt gia đình bạn gái.

**Nước hoa Chanel Eau de Parfum (Mã SP: CH123) – [GIÁ] VNĐ**
→ Nước hoa này sẽ làm nổi bật vẻ đẹp của bạn, mang đến một hương thơm quyến rũ và thanh lịch cho buổi gặp mặt gia đình.

Kết thúc bộ đồ với chiếc túi xách nhỏ gọn từ thương hiệu Coach (Mã SP: CO123) – [GIÁ] VNĐ, sẽ giúp bạn thêm tự tin và tiện lợi trong chuyến đi. Bộ outfit này mang đến vẻ đẹp tinh tế và thanh lịch cho buổi ra mắt gia đình bạn gái.

Bộ đồ tổng thể của bạn sẽ tạo nên một phong cách nữ tính nhưng vẫn giữ được sự chỉn chu và tin cậy, phù hợp với buổi gặp mặt gia đình bạn gái.

⏱️  [THỐNG KÊ TỐC ĐỘ]:
   - Nhả chữ đầu

In [14]:
# ═══════════════════════════════════════════════════════════════════
# HÀM TRUNG TÂM: Kết nối Layer B → Layer A → xây dựng context cho LLM
# ═══════════════════════════════════════════════════════════════════

def build_outfit_context(user_query: str, gender: str = "female") -> str:
    """
    Toàn bộ luồng Layer B:
      1. Tìm rule phối đồ cho sản phẩm user đang hỏi (Layer B lần 1)
      2. Tìm rule chi tiết cho từng món đồ gợi ý    (Layer B lần 2)
      3. Tìm sản phẩm thật trong Qdrant              (Layer A)
      4. Trả về chuỗi context để nhúng vào OUTFIT_SYSTEM_PROMPT

    Nếu không tìm thấy rule → trả về chuỗi rỗng để main loop fallback.
    """
    # ── Bước 1: Layer B lần 1 ─────────────────────────────────────
    base_rule = find_matching_rule(user_query, gender)
    if not base_rule:
        return ""   # Không tìm được → fallback về pipeline search

    # ── Bước 2: Layer B lần 2 ─────────────────────────────────────
    outfit_rules = find_outfit_details(base_rule, gender)
    if not outfit_rules:
        return ""

    # ── Bước 3: Layer A ───────────────────────────────────────────
    outfit_products = {}
    for layer_b_category, rule in outfit_rules.items():
        product_type = rule["rule_key"].split("|")[1].strip()
        products     = get_products_for_outfit(
            product_type    = product_type,
            layer_b_category= layer_b_category,
            phong_cach      = base_rule["phong_cach"],
            vector_db       = vector_db
        )
        outfit_products[layer_b_category] = {
            "product_type": product_type,
            "ly_do"       : rule["ly_do_tu_van"],
            "products"    : products,
        }

    # ── Bước 4: Build context string ─────────────────────────────
    lines = []
    lines.append("CÔNG THỨC PHỐI ĐỒ:")
    lines.append(f"  Phong cách : {base_rule['phong_cach']}")
    lines.append(f"  Bối cảnh   : {base_rule['boi_canh']}")
    lines.append(f"  Lý do chính: {base_rule['ly_do_tu_van']}")
    lines.append("")
    lines.append("SẢN PHẨM GỢI Ý:")

    for cat, data in outfit_products.items():
        lines.append(f"\n[{cat} – {data['product_type']}]")
        lines.append(f"  Lý do chọn: {data['ly_do']}")
        if data["products"]:
            for doc in data["products"]:
                pid   = doc.metadata.get("product_id", "N/A")
                price = doc.metadata.get("price", "N/A")
                lines.append(f"  • (Mã SP: {pid} | Giá: {price} VNĐ)")
                lines.append(f"    {doc.page_content[:200]}")
        else:
            lines.append("  • (Hiện tại chưa có sản phẩm phù hợp trong kho)")

    return "\n".join(lines)


# ── Kiểm tra nhanh (không cần chạy toàn bộ pipeline) ─────────────
print("Hàm build_outfit_context đã sẵn sàng.")
print("Chạy Cell 8 (Luồng chạy chính) để bắt đầu chat.")


Hàm build_outfit_context đã sẵn sàng.
Chạy Cell 8 (Luồng chạy chính) để bắt đầu chat.
